In [ ]:
!ls

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import fbeta_score
from datetime import datetime
from sklearn.model_selection import StratifiedKFold

In [ ]:
sample_df = pd.read_csv("/kaggle/input/competitions/super-ai-engineer-ss-6-heart-disease-prediction/sample_submission.csv")
test_df = pd.read_csv("/kaggle/input/competitions/super-ai-engineer-ss-6-heart-disease-prediction/test.csv")
train_df = pd.read_csv("/kaggle/input/competitions/super-ai-engineer-ss-6-heart-disease-prediction/train.csv")

In [ ]:
sample_df

In [ ]:
train_df

In [ ]:
test_df

In [ ]:
train_df.info()

In [ ]:
test_df.info()

In [ ]:
train_df.columns

In [ ]:
for i in train_df.columns:
    if i == 'ID':
        continue
    print(i, ':', train_df[i].unique())

In [ ]:
for col in train_df.columns:
    if train_df[col].dtype == 'object':  
        train_df[col] = train_df[col].fillna(method='ffill').fillna(method='bfill')
    else:
        train_df[col] = train_df[col].fillna(train_df[col].mean())

In [ ]:
train_df.info()

In [ ]:
train_df

In [ ]:
for i in train_df.columns:
    if i == 'ID':
        continue
    print(i, ':', train_df[i].unique())

In [ ]:
test_df['History of HeartDisease or Attack'] = 'No'
test_df['History of HeartDisease or Attack']

In [ ]:
binary_cols = [
    'History of HeartDisease or Attack',
    'High Blood Pressure',
    'Told High Cholesterol',
    'Cholesterol Checked',
    'Smoked 100+ Cigarettes',
    'Diagnosed Stroke',
    'Diagnosed Diabetes',
    'Leisure Physical Activity',
    'Heavy Alcohol Consumption',
    'Health Care Coverage',
    'Doctor Visit Cost Barrier',
    'Difficulty Walking',
    'Vegetable or Fruit Intake (1+ per Day)'
]

for col in binary_cols:
    train_df[col] = train_df[col].map({'No': 0, 'Yes': 1})
    test_df[col] = test_df[col].map({'No': 0, 'Yes': 1})

In [ ]:
train_df['Sex'] = train_df['Sex'].map({'Female': 0, 'Male': 1})
test_df['Sex'] = test_df['Sex'].map({'Female': 0, 'Male': 1})

In [ ]:
health_map = {
    'Very Poor': 0,
    'Poor': 1,
    'Fair': 2,
    'Good': 3,
    'Excellent': 4
}
train_df['General Health'] = train_df['General Health'].map(health_map)
test_df['General Health'] = test_df['General Health'].map(health_map)

In [ ]:
edu_map = {
    'Never attended school': 0,
    'Elementary': 1,
    'Some high school': 2,
    'High school graduate': 3,
    'Some college or technical school': 4,
    'College graduate': 5
}
train_df['Education Level'] = train_df['Education Level'].map(edu_map)
test_df['Education Level'] = test_df['Education Level'].map(edu_map)

In [ ]:
income_map = {
    'Less than $10,000': 0,
    '($10,000 to less than $15,000': 1,
    '$15,000 to less than $20,000': 2,
    '$20,000 to less than $25,000': 3,
    '$25,000 to less than $35,000': 4,
    '$35,000 to less than $50,000': 5,
    '$50,000 to less than $75,000': 6,
    '$75,000 or more': 7
}
train_df['Income Level'] = train_df['Income Level'].map(income_map)
test_df['Income Level'] = test_df['Income Level'].map(income_map)

In [ ]:
print(train_df.dtypes)

In [ ]:
for i in train_df.columns:
    if i == 'ID':
        continue
    print(i, ':', train_df[i].unique())

In [ ]:
for i in test_df.columns:
    if i == 'ID':
        continue
    print(i, ':', test_df[i].unique())

In [ ]:
train_df.describe()

In [ ]:
test_df.describe()

In [ ]:
X = train_df.drop(columns=['ID', 'History of HeartDisease or Attack'])
y = train_df['History of HeartDisease or Attack']

X_test = test_df.drop(columns=['ID', 'History of HeartDisease or Attack'], errors='ignore')

def add_features(df):
    df = df.copy()
    df['BMI_Age'] = df['Body Mass Index'] * df['Age']
    df['BP_Chol'] = df['High Blood Pressure'] * df['Told High Cholesterol']
    df['Inactive_Obese'] = ((df['Leisure Physical Activity'] == 0) & (df['Body Mass Index'] > 30)).astype(int)
    return df

X = add_features(X)
X_test = add_features(X_test)

X_test = X_test[X.columns]

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

thresholds = []
scores = []

for train_idx, val_idx in kf.split(X, y):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

    neg, pos = np.bincount(y_tr)
    scale_pos_weight = neg / pos

    model = XGBClassifier(
        n_estimators=1000,
        max_depth=4,
        learning_rate=0.02,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=5,
        gamma=1,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        tree_method='gpu_hist',
        random_state=42
    )

    model.fit(X_tr, y_tr)

    y_prob = model.predict_proba(X_va)[:, 1]

    best_f2, best_t = 0, 0.5
    for t in np.linspace(0.01, 0.5, 100):
        y_pred = (y_prob >= t).astype(int)
        f2 = fbeta_score(y_va, y_pred, beta=2)
        if f2 > best_f2:
            best_f2, best_t = f2, t

    thresholds.append(best_t)
    scores.append(best_f2)

print("CV Mean F2:", np.mean(scores))
best_threshold = np.mean(thresholds)
print("Best Threshold:", best_threshold)

neg, pos = np.bincount(y)
scale_pos_weight = neg / pos

final_model = XGBClassifier(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.02,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=5,
    gamma=1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    tree_method='hist',
    random_state=42
)

final_model.fit(X, y)

y_test_prob = final_model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= best_threshold).astype(int)

y_test_label = pd.Series(y_test_pred).map({0: 'No', 1: 'Yes'})

submission = pd.DataFrame({
    'ID': test_df['ID'],
    'History of HeartDisease or Attack': y_test_label
})

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'submission_{timestamp}.csv'

submission.to_csv(filename, index=False)

print(f"Saved: {filename}")